

The goal of this notebook is to evaluate how effectively **Deep Kernel Learning (DKL)** models can predict **ISUP prostate cancer grade** using **radiomics features** extracted from prostate MRI.

To achieve this, we systematically explore several modeling dimensions:

- **Feature subsets**
  - Top 10 most correlated features  
  - Top 20 most correlated features  
  - All available radiomics features  

- **Neural feature extractors**
  - **Large** extractor (deeper, higher capacity)  
  - **Small** extractor (lighter, more compact)  

- **Latent dimensions** for the Gaussian Process kernel  
  - 5, 10, 15, 20  

The notebook trains and evaluates DKL models across all combinations of these settings and reports performance using **MSE** and **R²**.  
The final output identifies the **best-performing model configuration**.


In [1]:

import torch
import gpytorch
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from deep_gp.test_dkl_class import LargeFeatureExtractor,SmallFeatureExtractor, GPRegressionModel

%matplotlib inline
%load_ext autoreload
%autoreload 2



In [2]:

features = pd.read_csv('../features.csv')
targets = pd.read_csv('../targets.csv')
data = features.merge(targets[['study_id','patient_id','case_ISUP']], on='study_id')
X = data.drop(['study_id','case_ISUP'], axis=1) # predictors
y = data['case_ISUP']  # target variable

In [3]:
df = X.copy() 
df["case_ISUP"] = y

idx_class0 = df.index[df["case_ISUP"] == 0]

# Storage for selected class-0 samples 
selected_class0 = []

# For each class 1–5, find nearest class-0 sample

for cls in range(1, 6): 
    print(f"Processing class {cls}") 
    idx_cls = df.index[df["case_ISUP"] == cls] 
    
    # Fit KNN on class 0 samples 
    knn = NearestNeighbors(n_neighbors=1) 
    knn.fit(df.loc[idx_class0, X.columns]) 
    
    # For each sample in class cls, find nearest class-0 sample 
    distances, indices = knn.kneighbors(df.loc[idx_cls, X.columns]) 
    # Convert neighbor indices to original dataframe indices 
    nearest_idx0 = idx_class0[indices.flatten()] 
    # Add to list 
    selected_class0.extend(nearest_idx0)

# keep only unique samples in class 0
selected_class0 = list(set(selected_class0)) 
print(f"\nOriginal class 0 count: {len(idx_class0)}") 
print(f"New class 0 count: {len(selected_class0)}")

# new reduced dataset
df_new = pd.concat([ df[df["case_ISUP"] != 0], # keep all minority classes 
                    df.loc[selected_class0] # keep only selected class-0 samples ])
])

print("\nFinal class counts:")
print(df_new["case_ISUP"].value_counts().sort_index())



# SMOTE only for classes 3, 4, 5

X_smote = df_new.drop(columns=["case_ISUP"])
y_smote = df_new["case_ISUP"]

sampling_strategy = {
    3: 150,
    4: 150,
    5: 150
}

sm = SMOTE(
    sampling_strategy=sampling_strategy,
    k_neighbors=3,
    random_state=42
)

X_resampled, y_resampled = sm.fit_resample(X_smote, y_smote)

df_resampled = pd.concat([
    pd.DataFrame(X_resampled, columns=X_smote.columns),
    pd.Series(y_resampled, name="case_ISUP")
], axis=1)

print("\nFinal class counts AFTER SMOTE:")
print(df_resampled["case_ISUP"].value_counts().sort_index())


Processing class 1
Processing class 2
Processing class 3
Processing class 4
Processing class 5

Original class 0 count: 589
New class 0 count: 262

Final class counts:
case_ISUP
0    262
1    157
2    154
3     69
4     27
5     35
Name: count, dtype: int64

Final class counts AFTER SMOTE:
case_ISUP
0    262
1    157
2    154
3    150
4    150
5    150
Name: count, dtype: int64


In [4]:
corr = df_resampled.corr(method="spearman")["case_ISUP"].abs().sort_values(ascending=False)
corr = corr.drop("case_ISUP")

top10 = corr.head(10).index.tolist()
top20 = corr.head(20).index.tolist()
all_features = df_resampled.drop(columns=["case_ISUP"]).columns.tolist()

print("Top 10:", top10)
print("Top 20:", top20)
print("Total features:", len(all_features))


Top 10: ['original_firstorder_Minimum', 'original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_gldm_DependenceEntropy', 'original_ngtdm_Strength', 'original_shape_LeastAxisLength', 'original_ngtdm_Coarseness', 'original_glrlm_LongRunHighGrayLevelEmphasis', 'original_gldm_SmallDependenceLowGrayLevelEmphasis']
Top 20: ['original_firstorder_Minimum', 'original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_gldm_DependenceEntropy', 'original_ngtdm_Strength', 'original_shape_LeastAxisLength', 'original_ngtdm_Coarseness', 'original_glrlm_LongRunHighGrayLevelEmphasis', 'original_gldm_SmallDependenceLowGrayLevelEmphasis', 'original_glszm_SmallAreaLowGrayLevelEmphasis', 'original_glcm_Imc2', 'original_shape_MeshVolume', 'original_shape_VoxelVolume', 'original_shape_MinorAxisLength', 'original_shape_MajorAxisLength', 'original_glrlm_RunEntropy', 'ori

In [5]:
def run_dkl_experiment(feature_list, df, latent_dim=10, extractor_type="large", n_epochs=300):

    print(f"\nRunning DKL with {len(feature_list)} features, "
          f"latent_dim={latent_dim}, extractor={extractor_type}")

    X = df[feature_list]
    y = df["case_ISUP"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
    y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()

    train_x = torch.tensor(X_train_scaled, dtype=torch.float32)
    train_y = torch.tensor(y_train_scaled, dtype=torch.float32)
    test_x  = torch.tensor(X_test_scaled, dtype=torch.float32)
    test_y  = torch.tensor(y_test_scaled, dtype=torch.float32)

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = GPRegressionModel(
        train_x, train_y, likelihood,
        data_dim=train_x.shape[1],
        latent_dim=latent_dim,
        extractor_type=extractor_type
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    model.train()
    likelihood.train()

    for i in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        optimizer.step()

    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = likelihood(model(test_x))

    # Predictions
    y_pred = scaler_y.inverse_transform(preds.mean.cpu().numpy().reshape(-1,1)).ravel()
    y_true = scaler_y.inverse_transform(test_y.cpu().numpy().reshape(-1,1)).ravel()

    # compute predictive uncertainty
    f_std = preds.stddev.cpu().numpy().ravel()

    # Build uncertainty dataframe
    df_unc = pd.DataFrame({
        "true": y_true,
        "pred": y_pred,
        "std": f_std
    })
    df_unc["true_class"] = np.round(df_unc["true"]).astype(int)

    # Metrics
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)

    print(f"MSE={mse:.4f}, R²={r2:.4f}")

    return mse, r2, df_unc


In [6]:
feature_sets = {
    "top10": top10,
    "top20": top20,
    "all": all_features
}

extractors = ["large", "small"]
latent_dims = [5, 10, 15, 20]

results = []

for feat_name, feat_list in feature_sets.items():
    print(f"\n==============================")
    print(f"   Testing feature set: {feat_name}")
    print(f"==============================")

    for ext in extractors:
        for ld in latent_dims:

            print(f"\n--- Running: features={feat_name}, extractor={ext}, latent_dim={ld} ---")

            
            mse, r2, df_unc = run_dkl_experiment(
                feature_list=feat_list,
                df=df_resampled,
                latent_dim=ld,
                extractor_type=ext
            )

        
            results.append({
                "features": feat_name,
                "extractor": ext,
                "latent_dim": ld,
                "mse": mse,
                "r2": r2,
                "uncertainty": df_unc
            })



   Testing feature set: top10

--- Running: features=top10, extractor=large, latent_dim=5 ---

Running DKL with 10 features, latent_dim=5, extractor=large
MSE=2.3486, R²=0.2638

--- Running: features=top10, extractor=large, latent_dim=10 ---

Running DKL with 10 features, latent_dim=10, extractor=large
MSE=2.3141, R²=0.2746

--- Running: features=top10, extractor=large, latent_dim=15 ---

Running DKL with 10 features, latent_dim=15, extractor=large
MSE=2.2555, R²=0.2930

--- Running: features=top10, extractor=large, latent_dim=20 ---

Running DKL with 10 features, latent_dim=20, extractor=large
MSE=2.1800, R²=0.3167

--- Running: features=top10, extractor=small, latent_dim=5 ---

Running DKL with 10 features, latent_dim=5, extractor=small
MSE=2.0296, R²=0.3638

--- Running: features=top10, extractor=small, latent_dim=10 ---

Running DKL with 10 features, latent_dim=10, extractor=small
MSE=2.1420, R²=0.3286

--- Running: features=top10, extractor=small, latent_dim=15 ---

Running DKL w

In [9]:
results_sorted = sorted(results, key=lambda x: x["r2"], reverse=True)
best = results_sorted[0]

print("\n===== BEST MODEL FOUND =====")
print(f"Feature set:   {best['features']}")
print(f"Extractor:     {best['extractor']}")
print(f"Latent dim:    {best['latent_dim']}")
print(f"MSE:           {best['mse']:.4f}")
print(f"R²:            {best['r2']:.4f}")

df_unc_best = best["uncertainty"]

unc_by_class = df_unc_best.groupby("true_class")["std"].mean()

print("\n===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====")
for cls, std in unc_by_class.items():
    print(f"ISUP {cls}:  std = {std:.4f}")



===== BEST MODEL FOUND =====
Feature set:   all
Extractor:     small
Latent dim:    20
MSE:           1.3201
R²:            0.5862

===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====
ISUP 0:  std = 0.2391
ISUP 1:  std = 0.2332
ISUP 2:  std = 0.2414
ISUP 3:  std = 0.2438
ISUP 4:  std = 0.2355
ISUP 5:  std = 0.2270
